In [55]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit, broadcast,countDistinct, count
spark = SparkSession.builder.appName("homework").getOrCreate()

In [2]:
#  - Disabled automatic broadcast join with `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")`
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [4]:
#   - Explicitly broadcast JOINs `medals` and `maps`

df_medals = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/medals.csv") \
            .withColumnRenamed('name', 'medals_name') \
            .withColumnRenamed('description', 'medals_description')    
        
df_maps = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/maps.csv") \
            .withColumnRenamed('name', 'map_name') \
            .withColumnRenamed('description', 'map_description') \
            .withColumnRenamed('mapid', 'mapid')

df_medals_maps_broadcast = df_medals.join(broadcast(df_maps))

df_medals_maps_broadcast.count()

7320

In [6]:
display(df_medals.count())
display(df_maps.count())

183

40

In [7]:
#   - Bucket join `match_details`, `matches`, and `medal_matches_players` on `match_id` with `16` buckets

df_match_details = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/match_details.csv")

df_matches = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/matches.csv")

df_medal_matches_players = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/medals_matches_players.csv")

In [8]:
display(df_match_details.count())
display(df_matches.count())
display(df_medal_matches_players.count())

151761

24025

755229

In [21]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog('default')
catalog.purge_table('bootcamp.match_details_bucketed')
catalog.purge_table('bootcamp.matches_bucketed')
catalog.purge_table('bootcamp.medal_matches_players_bucketed')

In [24]:
spark.sql(""" CREATE DATABASE IF NOT EXISTS bootcamp""")

num_buckets = 16

In [25]:
spark.catalog.listDatabases()

[Database(name='bootcamp', catalog='demo', description=None, locationUri='s3://warehouse/bootcamp')]

In [26]:
spark.catalog.listTables('bootcamp')

[]

In [27]:
spark.sql(""" drop table if exists bootcamp.match_details_bucketed """)

bucketed_match_details_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
    match_id STRING
    ,player_gamertag STRING
    ,previous_spartan_rank int
    ,spartan_rank int
    ,previous_total_xp int
    ,total_xp int
    ,previous_csr_tier int
    ,previous_csr_designation int
    ,previous_csr int
    ,previous_csr_percent_to_next_tier int
    ,previous_csr_rank int
    ,current_csr_tier int
    ,current_csr_designation int
    ,current_csr int
    ,current_csr_percent_to_next_tier int
    ,current_csr_rank int
    ,player_rank_on_team int
    ,player_finished boolean
    ,player_average_life int
    ,player_total_kills int
    ,player_total_headshots int
    ,player_total_weapon_damage int
    ,player_total_shots_landed int
    ,player_total_melee_kills int
    ,player_total_melee_damage int
    ,player_total_assassinations int
    ,player_total_ground_pound_kills int
    ,player_total_shoulder_bash_kills int
    ,player_total_grenade_damage int
    ,player_total_power_weapon_damage int
    ,player_total_power_weapon_grabs int
    ,player_total_deaths int
    ,player_total_assists int
    ,player_total_grenade_kills int
    ,did_win int
    ,team_id int
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_match_details_DDL)

df_match_details.write.bucketBy(num_buckets, "match_id").saveAsTable("bootcamp.match_details_bucketed", format="parquet", mode="overwrite")

In [28]:
spark.sql(""" drop table if exists bootcamp.matches_bucketed """)

bucketed_matches_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
    match_id STRING
    ,mapid STRING
    ,is_team_game boolean
    ,playlist_id STRING
    ,game_variant_id STRING
    ,is_match_over boolean
    ,completion_date timestamp
    ,match_duration STRING
    ,game_mode STRING
    ,map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_matches_DDL)

df_matches.write.bucketBy(16, "match_id").saveAsTable("bootcamp.matches_bucketed", format="parquet", mode="overwrite")

In [29]:
spark.sql(""" drop table if exists bootcamp.medal_matches_players_bucketed """)

bucketed_medal_matches_players_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.medal_matches_players_bucketed (
   match_id STRING,
   player_gamertag STRING,
   medal_id INTEGER,
   count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_medal_matches_players_DDL)

df_medal_matches_players.write.bucketBy(num_buckets, "match_id").saveAsTable("bootcamp.medal_matches_players_bucketed", format="parquet", mode="overwrite")

In [30]:
bucketed_match_details = spark.table("bootcamp.match_details_bucketed")
bucketed_matches = spark.table("bootcamp.matches_bucketed")
bucketed_medal_matches_players = spark.table("bootcamp.medal_matches_players_bucketed")

In [31]:
display(bucketed_match_details.count())
display(bucketed_matches.count())
display(bucketed_medal_matches_players.count())

151761

24025

755229

In [58]:

# Read df2 and join with bucketed df1
result = (bucketed_match_details.join(bucketed_matches, "match_id")
            .join(bucketed_medal_matches_players, "match_id")
            .join(df_medals, "medal_id")
            .join(df_maps, "mapid")          
            .select(bucketed_match_details["*"], bucketed_matches["mapid"], bucketed_matches["is_team_game"], bucketed_matches["playlist_id"]
                , bucketed_matches["game_variant_id"], bucketed_matches["is_match_over"], bucketed_matches["completion_date"]
                , bucketed_matches["match_duration"], bucketed_matches["game_mode"], bucketed_matches["map_variant_id"]
                , bucketed_medal_matches_players["medal_id"], bucketed_medal_matches_players["count"]
                , df_maps["map_name"], df_maps["map_description"]
                , df_medals["classification"], df_medals["difficulty"]
                , df_medals["medals_description"], df_medals["medals_name"]
                #, df_medals["sprite_height"], df_medals["sprite_left"], df_medals["sprite_sheet_height"]
                #, df_medals["sprite_sheet_width"], df_medals["sprite_top"], df_medals["sprite_uri"], df_medals["sprite_width"]
            )
         )

        # .join(matchDetailsBucketTable, Seq("match_id"))
        # .join(medalMatchesPlayersExpandedBucketTable, Seq("match_id"), "left")

In [53]:
display('match_details count is ' + str(bucketed_match_details.count()))
bucketed_match_details.select(countDistinct("match_id")).show()
display('match count is ' + str(bucketed_matches.count()))
bucketed_matches.select(countDistinct("match_id")).show()
display('medal_matches_players count is ' + str(bucketed_medal_matches_players.count()))
bucketed_medal_matches_players.select(countDistinct("match_id")).show()
display('after join count is ' + str(result.count()))
result.select(countDistinct("match_id")).show()

'match_details count is 151761'

+------------------------+
|count(DISTINCT match_id)|
+------------------------+
|                   19050|
+------------------------+



'match count is 24025'

+------------------------+
|count(DISTINCT match_id)|
+------------------------+
|                   24025|
+------------------------+



'medal_matches_players count is 755229'

+------------------------+
|count(DISTINCT match_id)|
+------------------------+
|                   18942|
+------------------------+



'after join count is 6885858'

24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:00:31 WARN RowBasedKeyValueBatch: Calling spill() on

+------------------------+
|count(DISTINCT match_id)|
+------------------------+
|                   18942|
+------------------------+



In [60]:
from pyspark.sql.functions import avg
#   - Aggregate the joined data frame to figure out questions like:
#     - Which player averages the most kills per game?

#group_data = result.groupBy(["player_gamertag", "match_id"])
#group_data.agg({'player_total_kills':'avg'}).show()

avg_kills = result.groupBy(["player_gamertag", "match_id"]).agg(avg("player_total_kills").alias("avg_kills"))

# most_kills = avg_kills.agg({"avg_kills": "max"}).collect()[0][0]
most_kills = avg_kills.orderBy("avg_kills", ascending=False).limit(1)

most_kills.show()

#print("player averages the most kills per game:", most_kills)

24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:03:01 WARN RowBasedKeyValueBatch: Calling spill() on

+---------------+--------------------+---------+
|player_gamertag|            match_id|avg_kills|
+---------------+--------------------+---------+
|   gimpinator14|acf0e47e-20ac-4b1...|    109.0|
+---------------+--------------------+---------+



In [63]:
from pyspark.sql.functions import count
#     - Which playlist gets played the most?

agg_playlist = result.groupBy(["playlist_id"]).agg(countDistinct("match_id").alias("agg_playlist_id"))

most_played = agg_playlist.orderBy("agg_playlist_id", ascending=False).limit(1)

most_played.show()

24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:05:10 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------+---------------+
|         playlist_id|agg_playlist_id|
+--------------------+---------------+
|f72e0ef0-7c4a-430...|           7640|
+--------------------+---------------+



In [65]:
from pyspark.sql.functions import count
#     - Which map gets played the most?

agg_map_id = result.groupBy(["mapid", "map_name"]).agg(countDistinct("match_id").alias("agg_mapid"))

most_map = agg_map_id.orderBy("agg_mapid", ascending=False).limit(1)

most_map.show()

24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/12/10 23:07:11 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------+--------------+---------+
|               mapid|      map_name|agg_mapid|
+--------------------+--------------+---------+
|c7edbf0f-f206-11e...|Breakout Arena|     7032|
+--------------------+--------------+---------+



In [66]:
from pyspark.sql.functions import sum
#     - Which map do players get the most Killing Spree medals on?
medal_map = (result.filter("classification == 'KillingSpree'")
             .groupBy(["mapid", "map_name"]).agg(countDistinct("match_id").alias("map_kills"))
             .select("map_name", "mapid","map_kills"))

most_kills_map = medal_map.orderBy("map_kills", ascending=False).limit(1)

most_kills_map.show()

+--------------+--------------------+---------+
|      map_name|               mapid|map_kills|
+--------------+--------------------+---------+
|Breakout Arena|c7edbf0f-f206-11e...|     4917|
+--------------+--------------------+---------+



In [67]:
medal_map.orderBy("map_kills", ascending=False).show()

+--------------+--------------------+---------+
|      map_name|               mapid|map_kills|
+--------------+--------------------+---------+
|Breakout Arena|c7edbf0f-f206-11e...|     4917|
|        Alpine|c74c9d0f-f206-11e...|     1290|
|        Empire|cdb934b0-f206-11e...|     1111|
|       The Rig|cb914b9e-f206-11e...|      917|
|         Truth|ce1dc2de-f206-11e...|      868|
|         Plaza|caacb800-f206-11e...|      829|
|      Coliseum|cebd854f-f206-11e...|      821|
|       Glacier|c7805740-f206-11e...|      814|
|        Regret|cdee4e70-f206-11e...|      809|
|          Eden|cd844200-f206-11e...|      788|
|        Fathom|cc040aa1-f206-11e...|      752|
|    Overgrowth|ca737f8f-f206-11e...|      495|
|      Riptide |cbcea2c0-f206-11e...|      417|
|          NULL|cc74f4e1-f206-11e...|      374|
|          NULL|ce89a40f-f206-11e...|      171|
|      Parallax|c7b7baf0-f206-11e...|      140|
+--------------+--------------------+---------+



In [20]:
#   - With the aggregated data set
#     - Try different `.sortWithinPartitions` to see which has the smallest data size (hint: playlists and maps are both very low cardinality)
result.rdd.getNumPartitions()

20

In [21]:
result.write.mode("overwrite").saveAsTable("bootcamp.resultUnsorted")

In [ ]:
result.sortWithinPartitions("playlist_id").write.mode("append").saveAsTable("bootcamp.sorted_by_playlist_id")

In [24]:
result.sortWithinPartitions("mapid").write.mode("append").saveAsTable("bootcamp.sorted_by_mapid")

In [26]:
%%sql
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'no sorting'
FROM bootcamp.resultUnsorted.files
UNION ALL
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'sorted_by_playlist_id'
FROM bootcamp.sorted_by_playlist_id.files
UNION ALL
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'sorted_by_mapid'
FROM bootcamp.sorted_by_mapid.files

total_size,number_of_files,no sorting
151626803,20,no sorting
147392783,20,sorted_by_playlist_id
150102931,20,sorted_by_mapid
